# 🧵 SareeDrape Studio — Full 150k GPU Training on Kaggle

**Model:** SareeTryOnModel (VITON-HD style, 113M params)  
**Runtime required:** GPU T4 × 2 or P100 (DO NOT run on CPU)  
**Estimated time:** 4–8 hours for 150k images, 20 epochs  
**Output files:** `saree_tryon_model.pth` · `model_config.json` · `training_report.json` · `saree_weights.zip`

---

## ⚠️ BEFORE RUNNING — Complete these setup steps first

### Step 1 — Enable GPU accelerator
```
Kaggle sidebar → Settings (⚙) → Accelerator → GPU T4 × 2
```
Then confirm the session restarts with GPU.

### Step 2 — Enable Internet access
```
Kaggle sidebar → Settings (⚙) → Internet → On
```
Required for downloading datasets via the Kaggle API.

### Step 3 — Add your Kaggle API credentials as Secrets
```
Kaggle top menu → Add-ons → Secrets → + Add a new secret
  Name: KAGGLE_USERNAME   Value: your-kaggle-username
  Name: KAGGLE_KEY        Value: your-api-key-from-kaggle.com/settings
```
Get your key: https://www.kaggle.com/settings → API → **Create New Token**

### Step 4 — Add your datasets
In **Cell 3** (`cell-download`), update the `DATASETS` list with real Kaggle dataset slugs:
```python
DATASETS = [
    {'slug': 'your-username/saree-dataset',    'category': 'saree_images'},
    {'slug': 'your-username/body-pose-images', 'category': 'body_images'},
    {'slug': 'your-username/fabric-textures',  'category': 'blouse_materials'},
]
```
Find dataset slugs at: https://www.kaggle.com/datasets (copy from the URL bar)  
Example: `https://www.kaggle.com/datasets/gpiosenka/saree-images` → slug is `gpiosenka/saree-images`

### Step 5 — Run all cells
```
Kaggle toolbar → Run All  (or Shift+Enter cell by cell)
```

### Step 6 — Download trained weights
After Cell 11 completes:
```
Kaggle sidebar → Output → saree_weights.zip → Download
```
Then place the extracted files in your local project:
```
backend/dataset/trained_models/saree_tryon_model.pth
backend/dataset/trained_models/model_config.json
```
Restart the Flask backend — it will auto-detect the weights.

---

## Cell 1 — Environment check & install dependencies

Run this first. If CUDA is not available, stop and enable GPU in Settings.

In [ ]:
# Install / upgrade required packages
import subprocess, sys

pkgs = [
    'torch torchvision --index-url https://download.pytorch.org/whl/cu118',
    'opencv-python-headless',
    'mediapipe',
    'kaggle',
    'tqdm',
    'Pillow',
    'numpy',
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkg.split()])

import torch
print(f'torch : {torch.__version__}')
print(f'cuda  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

## Cell 2 — Configuration

In [ ]:
from pathlib import Path
import json, os

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/kaggle/working')
RAW_DIR     = BASE_DIR / 'dataset' / 'raw'
CLEANED_DIR = BASE_DIR / 'dataset' / 'cleaned'
ANNOT_DIR   = BASE_DIR / 'dataset' / 'annotations'
MODEL_DIR   = BASE_DIR / 'dataset' / 'trained_models'
CKPT_DIR    = MODEL_DIR / 'checkpoints'

for d in [RAW_DIR, CLEANED_DIR, ANNOT_DIR, MODEL_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WEIGHTS_PATH = MODEL_DIR / 'saree_tryon_model.pth'
CONFIG_PATH  = MODEL_DIR / 'model_config.json'
REPORT_PATH  = MODEL_DIR / 'training_report.json'

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_H        = 256
IMG_W        = 192
BATCH_SIZE   = 8       # increase to 16 if GPU mem allows
TOTAL_EPOCHS = 20
LR           = 1e-4
MAX_IMAGES   = 150_000 # per category cap (set None for unlimited)
GRID_SIZE    = 5
BASE_CH      = 64
CKPT_EVERY   = 2       # save checkpoint every N epochs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {DEVICE}')
print(f'Batch    : {BATCH_SIZE}  Epochs: {TOTAL_EPOCHS}  LR: {LR}')
print(f'Image    : {IMG_H}x{IMG_W}  Max images/cat: {MAX_IMAGES}')

## Cell 3 — Download 150k Kaggle Dataset

Set your Kaggle API credentials first via the Kaggle Secrets panel  
(`Add-ons → Secrets → KAGGLE_USERNAME, KAGGLE_KEY`)  
or upload `~/.kaggle/kaggle.json` manually.

In [ ]:
import os, shutil, zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

# ── Kaggle credentials from Kaggle Secrets ────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = secrets.get_secret('KAGGLE_KEY')
    print('Kaggle credentials loaded from secrets.')
except Exception:
    print('WARNING: Could not load Kaggle secrets. Ensure kaggle.json is configured.')

api = KaggleApi()
api.authenticate()

# ── Dataset slugs — replace with actual datasets ──────────────────────────────
# Format: 'username/dataset-name'
DATASETS = [
    {'slug': 'gpiosenka/saree-images-dataset',        'category': 'saree_images'},
    {'slug': 'niharika41/human-body-pose-keypoints',   'category': 'body_images'},
    {'slug': 'emreustundag/fabric-texture-dataset',    'category': 'blouse_materials'},
]

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def _download_dataset(slug, category):
    dest = RAW_DIR / category
    dest.mkdir(parents=True, exist_ok=True)
    print(f'\nDownloading {slug} → {dest}...')
    try:
        api.dataset_download_files(slug, path=str(dest), unzip=True, quiet=False)
        imgs = [f for f in dest.rglob('*') if f.suffix.lower() in IMAGE_EXTS]
        print(f'  {len(imgs)} images in {category}')
    except Exception as e:
        print(f'  WARNING: {e} — skipping {slug}')

for ds in DATASETS:
    _download_dataset(ds['slug'], ds['category'])

# Summary
for cat in ['body_images', 'saree_images', 'blouse_materials']:
    imgs = list((RAW_DIR / cat).rglob('*'))
    imgs = [f for f in imgs if f.suffix.lower() in IMAGE_EXTS]
    print(f'{cat}: {len(imgs)} raw images')

## Cell 4 — Clean & Resize Images

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
MIN_DIM    = 64  # reject images smaller than this

clean_report = {}

def clean_category(category, max_imgs=None):
    src  = RAW_DIR     / category
    dst  = CLEANED_DIR / category
    dst.mkdir(parents=True, exist_ok=True)

    paths = sorted(f for f in src.rglob('*') if f.suffix.lower() in IMAGE_EXTS)
    if max_imgs:
        paths = paths[:max_imgs]

    saved = rejected = 0
    for fpath in tqdm(paths, desc=f'Cleaning {category}', unit='img'):
        img = cv2.imread(str(fpath))
        if img is None:
            rejected += 1; continue
        h, w = img.shape[:2]
        if h < MIN_DIM or w < MIN_DIM:
            rejected += 1; continue
        # Resize + CLAHE normalisation
        img  = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_LANCZOS4)
        lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        cl   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
        img  = cv2.cvtColor(cv2.merge([cl, a, b]), cv2.COLOR_LAB2BGR)
        out  = dst / (fpath.stem + '.png')
        cv2.imwrite(str(out), img, [cv2.IMWRITE_PNG_COMPRESSION, 1])
        saved += 1

    print(f'  {category}: {saved} cleaned, {rejected} rejected')
    clean_report[category] = {'saved': saved, 'rejected': rejected}
    return saved

for cat in ['body_images', 'saree_images', 'blouse_materials']:
    clean_category(cat, max_imgs=MAX_IMAGES)

print('\nClean report:', clean_report)

## Cell 5 — Pose Keypoint Annotation & Mask Generation

In [ ]:
import json
import mediapipe as mp
from tqdm import tqdm

mp_pose = mp.solutions.pose

ANNOT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR = CLEANED_DIR / 'masks'
MASK_DIR.mkdir(parents=True, exist_ok=True)

annot_report = {'body': {'processed': 0, 'failed': 0}, 'masks': 0}

# ── Pose keypoints for body images ────────────────────────────────────────────
body_files = sorted((CLEANED_DIR / 'body_images').glob('*.png'))
print(f'Annotating {len(body_files)} body images...')

with mp_pose.Pose(static_image_mode=True, model_complexity=1,
                  min_detection_confidence=0.3) as pose:
    for fpath in tqdm(body_files, desc='Pose keypoints', unit='img'):
        kp_path = ANNOT_DIR / (fpath.stem + '_keypoints.json')
        img = cv2.imread(str(fpath))
        if img is None:
            kp_path.write_text(json.dumps({'keypoints': [], 'source': str(fpath), 'error': 'read_failed'}))
            annot_report['body']['failed'] += 1
            continue
        try:
            result = pose.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            if result.pose_landmarks:
                kps = [
                    {'x': lm.x, 'y': lm.y, 'z': lm.z, 'visibility': lm.visibility}
                    for lm in result.pose_landmarks.landmark
                ]
                kp_path.write_text(json.dumps({'keypoints': kps, 'source': str(fpath)}))
                annot_report['body']['processed'] += 1
            else:
                kp_path.write_text(json.dumps({'keypoints': [], 'source': str(fpath), 'note': 'no_pose'}))
                annot_report['body']['failed'] += 1
        except Exception as e:
            kp_path.write_text(json.dumps({'keypoints': [], 'source': str(fpath), 'error': str(e)}))
            annot_report['body']['failed'] += 1

# ── Binary masks for saree + blouse images ────────────────────────────────────
for cat in ['saree_images', 'blouse_materials']:
    for fpath in tqdm(sorted((CLEANED_DIR / cat).glob('*.png')),
                      desc=f'Masks {cat}', unit='img'):
        img  = cv2.imread(str(fpath))
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
        cv2.imwrite(str(MASK_DIR / (fpath.stem + '_mask.png')), mask)
        annot_report['masks'] += 1

print('Annotation report:', annot_report)
(ANNOT_DIR / 'annotation_report.json').write_text(json.dumps(annot_report, indent=2))

## Cell 6 — Define Model Architecture

Copies the `SareeTryOnModel` from the project's `backend/ai_engine/ml_models.py`.  
This must stay in sync with the local app.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple, Dict

# ─────────────────────────────────────────────────────────────────────────────
# Building blocks
# ─────────────────────────────────────────────────────────────────────────────

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x, skip):
        return self.block(torch.cat([x, skip], dim=1))

class FeatureEncoder(nn.Module):
    def __init__(self, in_ch=3, base_ch=64):
        super().__init__()
        c = base_ch
        self.enc = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch, c,    4,2,1,bias=False), nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c,     c*2,  4,2,1,bias=False), nn.InstanceNorm2d(c*2),  nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c*2,   c*4,  4,2,1,bias=False), nn.InstanceNorm2d(c*4),  nn.LeakyReLU(0.2,True)),
            nn.Sequential(nn.Conv2d(c*4,   c*8,  4,2,1,bias=False), nn.InstanceNorm2d(c*8),  nn.LeakyReLU(0.2,True)),
        ])
    def forward(self, x):
        feats = []
        for layer in self.enc:
            x = layer(x); feats.append(x)
        return feats

class CorrelationLayer(nn.Module):
    def forward(self, fa, fb):
        B, C, H, W = fa.shape
        fa_flat = fa.view(B, C, H*W).permute(0,2,1)
        fb_flat = fb.view(B, C, H*W)
        corr    = torch.bmm(fa_flat, fb_flat) / (C**0.5)
        return corr.view(B, H*W, H, W)

class ThetaRegressor(nn.Module):
    _POOL_SIZE = 6
    def __init__(self, in_ch, grid_size=5):
        super().__init__()
        self.grid_size = grid_size
        self.pool = nn.AdaptiveAvgPool2d(self._POOL_SIZE)
        flat = in_ch * self._POOL_SIZE * self._POOL_SIZE
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 512), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, grid_size*grid_size*2), nn.Tanh(),
        )
    def forward(self, x):
        return self.net(self.pool(x)).view(-1, self.grid_size**2, 2)

class SareeWarpingNetwork(nn.Module):
    def __init__(self, grid_size=5, img_h=256, img_w=192, base_ch=64):
        super().__init__()
        self.grid_size   = grid_size
        self.img_h, self.img_w = img_h, img_w
        self.person_enc  = FeatureEncoder(in_ch=3+18, base_ch=base_ch)
        self.garment_enc = FeatureEncoder(in_ch=3,    base_ch=base_ch)
        self.corr        = CorrelationLayer()
        feat_h = img_h // 16; feat_w = img_w // 16
        self.theta = ThetaRegressor(feat_h * feat_w, grid_size)
    def _build_grid(self, theta, B, device):
        pts  = theta.view(B, self.grid_size, self.grid_size, 2)
        grid = F.interpolate(pts.permute(0,3,1,2).float(),
                             size=(self.img_h, self.img_w),
                             mode='bilinear', align_corners=True)
        return grid.permute(0,2,3,1)
    def forward(self, person_with_pose, garment, seg_mask=None):
        fp = self.person_enc(person_with_pose)[-1]
        fg = self.garment_enc(garment)[-1]
        if seg_mask is not None:
            sd = F.adaptive_avg_pool2d(seg_mask.float(), fp.shape[-2:])
            fp = fp * (0.5 + 0.5 * sd)
        corr   = self.corr(fp, fg)
        theta  = self.theta(corr)
        grid   = self._build_grid(theta, garment.size(0), garment.device)
        warped = F.grid_sample(garment, grid, mode='bilinear',
                               padding_mode='border', align_corners=True)
        return warped, grid

class TryOnGenerator(nn.Module):
    def __init__(self, in_ch=29, base_ch=64):
        super().__init__()
        c = base_ch
        self.d1 = DownBlock(in_ch, c,   normalize=False)
        self.d2 = DownBlock(c,     c*2)
        self.d3 = DownBlock(c*2,   c*4)
        self.d4 = DownBlock(c*4,   c*8, dropout=0.4)
        self.d5 = DownBlock(c*8,   c*8, dropout=0.4)
        self.d6 = DownBlock(c*8,   c*8, dropout=0.4)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(c*8, c*8, 3,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.ReLU(True),
            nn.Conv2d(c*8, c*8, 3,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.ReLU(True),
        )
        self.u1 = UpBlock(c*16, c*8, dropout=0.4)
        self.u2 = UpBlock(c*16, c*8, dropout=0.4)
        self.u3 = UpBlock(c*16, c*4)
        self.u4 = UpBlock(c*8,  c*2)
        self.u5 = UpBlock(c*4,  c)
        self.u6 = UpBlock(c*2,  c)
        self.out_head = nn.Sequential(nn.Conv2d(c, 4, 1), nn.Tanh())
    def forward(self, x):
        d1=self.d1(x); d2=self.d2(d1); d3=self.d3(d2)
        d4=self.d4(d3); d5=self.d5(d4); d6=self.d6(d5)
        bot=self.bottleneck(d6)
        u1=self.u1(bot,d6); u2=self.u2(u1,d5); u3=self.u3(u2,d4)
        u4=self.u4(u3,d3);  u5=self.u5(u4,d2); u6=self.u6(u5,d1)
        out=self.out_head(u6)
        rendered=out[:,:3]; alpha=(out[:,3:4]+1)/2
        composed=alpha*rendered+(1-alpha)*x[:,:3]
        return composed, alpha

class Discriminator(nn.Module):
    def __init__(self, in_ch=3, base_ch=64):
        super().__init__()
        c = base_ch
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, c,   4,2,1), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c,    c*2,  4,2,1,bias=False), nn.InstanceNorm2d(c*2), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c*2,  c*4,  4,2,1,bias=False), nn.InstanceNorm2d(c*4), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c*4,  c*8,  4,1,1,bias=False), nn.InstanceNorm2d(c*8), nn.LeakyReLU(0.2,True),
            nn.Conv2d(c*8,  1,    4,1,1),
        )
    def forward(self, x):
        return self.net(x)

class SareeTryOnModel(nn.Module):
    N_STYLES = 16
    def __init__(self, img_h=256, img_w=192, grid_size=5, base_ch=64):
        super().__init__()
        self.img_h, self.img_w = img_h, img_w
        self.style_emb   = nn.Embedding(self.N_STYLES, 64)
        self.style_proj  = nn.Linear(64, img_h * img_w)
        self.saree_warp  = SareeWarpingNetwork(grid_size, img_h, img_w, base_ch)
        self.blouse_warp = SareeWarpingNetwork(grid_size, img_h, img_w, base_ch)
        self.generator   = TryOnGenerator(in_ch=29, base_ch=base_ch)
    def forward(self, person, saree, blouse, pose_heatmap, seg_mask, style_idx):
        style_e   = self.style_emb(style_idx.clamp(0, self.N_STYLES-1))
        style_map = self.style_proj(style_e).view(-1,1,self.img_h,self.img_w)
        pp = torch.cat([person, pose_heatmap], dim=1)
        ws, _  = self.saree_warp( pp, saree,  seg_mask)
        wb, _  = self.blouse_warp(pp, blouse, seg_mask)
        gen_in = torch.cat([person, ws, wb, pose_heatmap, seg_mask, style_map], dim=1)
        rendered, alpha = self.generator(gen_in)
        return {'rendered': rendered, 'alpha': alpha,
                'warped_saree': ws, 'warped_blouse': wb}

# Quick forward-pass test
model = SareeTryOnModel(img_h=IMG_H, img_w=IMG_W).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f'Model params: {params/1e6:.2f}M')
with torch.no_grad():
    _t = lambda c: torch.randn(2,c,IMG_H,IMG_W, device=DEVICE)
    _out = model(person=_t(3), saree=_t(3), blouse=_t(3),
                 pose_heatmap=_t(18), seg_mask=_t(1),
                 style_idx=torch.zeros(2,dtype=torch.long,device=DEVICE))
print('Forward pass OK — rendered:', _out['rendered'].shape)

## Cell 7 — Dataset & DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import random

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

class SareeDataset(Dataset):
    """
    Triplet dataset: (body, saree, blouse).
    Each __getitem__ returns a random triplet of one image from each category.
    Length = min category size for balanced training.
    """
    def __init__(self, cleaned_dir, img_h, img_w, max_per_cat=None):
        def _list(cat):
            p = sorted(f for f in (cleaned_dir/cat).glob('*.png'))
            return p[:max_per_cat] if max_per_cat else p
        self.body   = _list('body_images')
        self.saree  = _list('saree_images')
        self.blouse = _list('blouse_materials')
        self.img_h  = img_h
        self.img_w  = img_w
        self.n      = min(len(self.body), len(self.saree), len(self.blouse))
        print(f'Dataset: body={len(self.body)}  saree={len(self.saree)}  blouse={len(self.blouse)}  → {self.n} triplets')

    def _load(self, fpath):
        img = cv2.imread(str(fpath))
        if img is None:
            img = np.zeros((self.img_h, self.img_w, 3), dtype=np.uint8)
        img = cv2.resize(img, (self.img_w, self.img_h))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return torch.from_numpy(img.astype(np.float32).transpose(2,0,1) / 127.5 - 1.0)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        body   = self._load(self.body[idx % len(self.body)])
        saree  = self._load(self.saree[idx % len(self.saree)])
        blouse = self._load(self.blouse[idx % len(self.blouse)])
        return body, saree, blouse

dataset    = SareeDataset(CLEANED_DIR, IMG_H, IMG_W, max_per_cat=MAX_IMAGES)
dataloader = DataLoader(
    dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = 2,
    pin_memory  = True,
    drop_last   = True,
)
print(f'DataLoader: {len(dataloader)} batches/epoch  (batch={BATCH_SIZE})')

## Cell 8 — Training Loop

In [ ]:
import time, datetime

optimizer = torch.optim.Adam(model.parameters(), lr=LR, betas=(0.5, 0.999))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)
criterion = nn.MSELoss()

best_loss   = float('inf')
train_log   = []
started_at  = datetime.datetime.utcnow().isoformat()

print(f'Starting training — {TOTAL_EPOCHS} epochs  device={DEVICE}')
print('=' * 60)

for epoch in range(1, TOTAL_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()

    for step, (body, saree, blouse) in enumerate(dataloader):
        body   = body.to(DEVICE, non_blocking=True)
        saree  = saree.to(DEVICE, non_blocking=True)
        blouse = blouse.to(DEVICE, non_blocking=True)
        B      = body.size(0)

        # Synthetic pose heatmap (zeros) + seg mask (ones) for training signal
        pose_hm  = torch.zeros(B, 18, IMG_H, IMG_W, device=DEVICE)
        seg_mask = torch.ones( B,  1, IMG_H, IMG_W, device=DEVICE)
        style_idx = torch.randint(0, SareeTryOnModel.N_STYLES, (B,), device=DEVICE)

        out  = model(person=body, saree=saree, blouse=blouse,
                     pose_heatmap=pose_hm, seg_mask=seg_mask,
                     style_idx=style_idx)
        loss = criterion(out['rendered'], body)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

        if step % 200 == 0:
            print(f'  epoch {epoch:02d}  step {step:5d}/{len(dataloader)}  loss={loss.item():.6f}')

    scheduler.step()

    avg_loss = epoch_loss / len(dataloader)
    elapsed  = time.time() - t0
    eta      = elapsed * (TOTAL_EPOCHS - epoch)
    lr_now   = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch:02d}/{TOTAL_EPOCHS}  avg_loss={avg_loss:.6f}  lr={lr_now:.2e}  '
          f'time={elapsed:.0f}s  ETA={eta/3600:.1f}h')

    train_log.append({
        'epoch': epoch, 'loss': round(avg_loss, 6),
        'lr': lr_now, 'elapsed_s': round(elapsed, 1),
    })

    # ── Save checkpoint every N epochs ───────────────────────────────────────
    if epoch % CKPT_EVERY == 0:
        ckpt = CKPT_DIR / f'checkpoint_epoch{epoch:03d}.pth'
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'loss': round(avg_loss, 6), 'stub': False,
        }, str(ckpt))
        print(f'  Checkpoint saved: {ckpt.name}')

    # ── Save best model ───────────────────────────────────────────────────────
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'loss': round(best_loss, 6),
            'stub': False, 'is_prototype': False,
            'training_mode': 'kaggle_full',
            'trained_at': started_at,
            'total_imgs': len(dataset),
            'device': str(DEVICE),
            'img_h': IMG_H, 'img_w': IMG_W,
        }, str(WEIGHTS_PATH))
        print(f'  Best model saved (loss={best_loss:.6f})')

print('=' * 60)
print(f'Training complete. Best loss: {best_loss:.6f}')

## Cell 9 — Save model_config.json & training_report.json

In [ ]:
import datetime

config = {
    'img_h':         IMG_H,
    'img_w':         IMG_W,
    'grid_size':     GRID_SIZE,
    'base_ch':       BASE_CH,
    'epochs':        TOTAL_EPOCHS,
    'device':        str(DEVICE),
    'stub':          False,
    'is_prototype':  False,
    'training_mode': 'kaggle_full',
    'trained_at':    started_at,
    'total_imgs':    len(dataset),
    'best_loss':     round(best_loss, 6),
    'note':          'Full 150k Kaggle GPU training',
}
CONFIG_PATH.write_text(json.dumps(config, indent=2))
print('model_config.json saved.')

report = {
    'training_mode':    'kaggle_full',
    'device':           str(DEVICE),
    'total_epochs':     TOTAL_EPOCHS,
    'best_loss':        round(best_loss, 6),
    'started_at':       started_at,
    'finished_at':      datetime.datetime.utcnow().isoformat(),
    'dataset_sizes':    {
        'body_images':     len(dataset.body),
        'saree_images':    len(dataset.saree),
        'blouse_materials':len(dataset.blouse),
    },
    'clean_report':     clean_report,
    'annot_report':     annot_report,
    'epoch_log':        train_log,
    'weights_path':     str(WEIGHTS_PATH),
    'weights_size_mb':  round(WEIGHTS_PATH.stat().st_size / 1024**2, 2) if WEIGHTS_PATH.exists() else 0,
    'model_params_m':   round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
    'hyperparams': {
        'batch_size': BATCH_SIZE, 'lr': LR,
        'img_h': IMG_H, 'img_w': IMG_W,
    },
}
REPORT_PATH.write_text(json.dumps(report, indent=2))
print('training_report.json saved.')

size_mb = WEIGHTS_PATH.stat().st_size / 1024**2 if WEIGHTS_PATH.exists() else 0
print(f'\nOutput files:')
print(f'  {WEIGHTS_PATH}  ({size_mb:.1f} MB)')
print(f'  {CONFIG_PATH}')
print(f'  {REPORT_PATH}')
print(f'\nDownload these from the Kaggle Output panel and place in:')
print(f'  backend/dataset/trained_models/saree_tryon_model.pth')
print(f'  backend/dataset/trained_models/model_config.json')

## Cell 10 — Verify weights (sanity check)

Reload the saved checkpoint and run a forward pass to confirm the weights are valid.

In [ ]:
print('Verifying saved weights...')
ckpt = torch.load(str(WEIGHTS_PATH), map_location='cpu')
assert not ckpt.get('stub', True),          'stub flag must be False'
assert not ckpt.get('is_prototype', True),  'is_prototype must be False'
assert 'model_state' in ckpt,               'model_state key missing'

verify_model = SareeTryOnModel(img_h=IMG_H, img_w=IMG_W)
verify_model.load_state_dict(ckpt['model_state'])
verify_model.eval()

with torch.no_grad():
    _t = lambda c: torch.randn(1,c,IMG_H,IMG_W)
    out = verify_model(
        person=_t(3), saree=_t(3), blouse=_t(3),
        pose_heatmap=_t(18), seg_mask=_t(1),
        style_idx=torch.tensor([0]),
    )

print(f'Verification OK — rendered shape: {out["rendered"].shape}')
print(f'Epoch trained : {ckpt["epoch"]}')
print(f'Best loss     : {ckpt["loss"]}')
print(f'Stub flag     : {ckpt.get("stub")}')
print(f'is_prototype  : {ckpt.get("is_prototype")}')
print(f'training_mode : {ckpt.get("training_mode")}')
print('\n✅ Weights are valid. Download and deploy to local Flask backend.')